In [14]:
# Install required packages (run once)
# Uncomment and run if packages are not already installed in your environment
!pip install ultralytics opencv-python pillow tqdm



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
# 1) Import required libraries
import os
import re
from pathlib import Path
from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm

# ultralytics YOLO (modern interface)
from ultralytics import YOLO


In [27]:
# 2) Paths and setup
BASE_DIR = Path('.')
FRAMES_FOLDER = BASE_DIR / 'saved_frames'
OUTPUT_FOLDER = BASE_DIR / 'cropped_persons'
MODEL_PATH = BASE_DIR / 'yolo11n.pt'

# Create output folder if it doesn't exist
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print('Frames folder:', FRAMES_FOLDER)
print('Output folder:', OUTPUT_FOLDER)
print('Model path:', MODEL_PATH)


Frames folder: saved_frames
Output folder: cropped_persons
Model path: yolo11n.pt


In [28]:
# 3) Load YOLO model
# This will load the local `yolo11n.pt` file.
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'yolo model not found at {MODEL_PATH}. Place yolo11n.pt in the same folder as this notebook.')

model = YOLO(str(MODEL_PATH))
print('Model loaded successfully')


Model loaded successfully


In [29]:
# Helper: extract frame number from filename like 'frame_012_00-00-05.jpg'
def extract_frame_number(filename):
    m = re.search(r'frame_(\d+)', filename)
    if m:
        return int(m.group(1))
    # fallback: try any digits in filename
    m2 = re.search(r'(\d+)', filename)
    return int(m2.group(1)) if m2 else None


In [30]:
# 4) Process frames: detect, crop, and save persons
CONF_THRESHOLD = 0.2  # only keep detections with confidence >= 0.2
PERSON_CLASS_ID = 0   # COCO class id for 'person' is 0
total_saved = 0

# get sorted list of image files in frames folder
if not FRAMES_FOLDER.exists():
    raise FileNotFoundError(f'saved_frames folder not found at {FRAMES_FOLDER}. Run frame extraction first.')

image_files = sorted([p for p in FRAMES_FOLDER.iterdir() if p.is_file() and p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f'Total frames found: {len(image_files)}')

for img_path in tqdm(image_files, desc='Processing frames'):
    try:
        img_bgr = cv2.imread(str(img_path))
        if img_bgr is None:
            print(f'Skipping unreadable image: {img_path.name}')
            continue

        # Make a copy for cropping (image in BGR from cv2)
        h, w = img_bgr.shape[:2]

        # Run detection. ultralytics returns a Results object list; take the first result.
        results = model(img_bgr, conf=CONF_THRESHOLD, imgsz=640)
        if len(results) == 0:
            print(f'No result for {img_path.name}')
            continue
        res = results[0]

        # boxes.data -> numpy array with columns [x1, y1, x2, y2, score, class]
        boxes = getattr(res, 'boxes', None)
        if boxes is None or len(boxes) == 0:
            print(f'No detections in {img_path.name}')
            continue

        # get box tensor data safely (works with ultralytics recent versions)
        try:
            box_data = boxes.data.cpu().numpy()  # shape (N,6)
        except Exception:
            # fallback: try extracting xyxy, conf, cls arrays
            xyxy = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            cls_ids = boxes.cls.cpu().numpy()
            box_data = []
            for (x1,y1,x2,y2),c,cl in zip(xyxy, confs, cls_ids):
                box_data.append([x1,y1,x2,y2,c,cl])
            box_data = np.array(box_data)

        saved_this_frame = 0
        frame_num = extract_frame_number(img_path.name) or img_path.stem

        for det_idx, row in enumerate(box_data, start=1):
            x1, y1, x2, y2, score, cls_id = row[:6]
            cls_id = int(cls_id)
            if cls_id != PERSON_CLASS_ID:
                continue  # skip non-person classes
            if score < CONF_THRESHOLD:
                continue  # skip low confidence (extra safety)

            # Convert to ints and clip to image bounds
            x1i = max(0, int(round(x1)))
            y1i = max(0, int(round(y1)))
            x2i = min(w, int(round(x2)))
            y2i = min(h, int(round(y2)))

            # Ensure valid box
            if x2i - x1i <= 5 or y2i - y1i <= 5:
                # skip tiny/invalid crops
                continue

            # Crop and save (convert BGR->RGB for PIL if desired, but cv2.imwrite accepts BGR)
            crop = img_bgr[y1i:y2i, x1i:x2i]
            if crop is None or crop.size == 0:
                continue

            saved_this_frame += 1
            total_saved += 1

            out_name = f'frame_{frame_num}_person_{saved_this_frame}.jpg'
            out_path = OUTPUT_FOLDER / out_name
            # write with cv2 (BGR)
            cv2.imwrite(str(out_path), crop)

        print(f'Saved {saved_this_frame} cropped persons from {img_path.name}')

    except Exception as e:
        print(f'Error processing {img_path.name}: {e}')

print('=== PROCESSING COMPLETED ===')
print(f'Total cropped persons saved: {total_saved}')


Total frames found: 85


Processing frames:   0%|          | 0/85 [00:00<?, ?it/s]


0: 640x384 1 person, 1 fire hydrant, 82.2ms
Speed: 3.1ms preprocess, 82.2ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 384)


Processing frames:   1%|          | 1/85 [00:00<00:14,  5.99it/s]

Saved 1 cropped persons from frame_001_00-00-00.jpg

0: 640x384 2 persons, 77.6ms
Speed: 2.8ms preprocess, 77.6ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 384)


...

No detections in frame_082_00-00-39.jpg

0: 640x384 (no detections), 78.9ms
Speed: 2.5ms preprocess, 78.9ms inference, 1.2ms postprocess per image at shape (1, 3, 640, 384)


Processing frames:  98%|█████████▊| 83/85 [00:10<00:00,  7.75it/s]

No detections in frame_083_00-00-40.jpg

0: 640x384 (no detections), 78.5ms
Speed: 2.5ms preprocess, 78.5ms inference, 1.0ms postprocess per image at shape (1, 3, 640, 384)
No detections in frame_084_00-00-40.jpg

0: 640x384 (no detections), 92.0ms
Speed: 2.5ms preprocess, 92.0ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 384)


Processing frames: 100%|██████████| 85/85 [00:10<00:00,  7.87it/s]

No detections in frame_085_00-00-41.jpg
=== PROCESSING COMPLETED ===
Total cropped persons saved: 73
